<a href="https://colab.research.google.com/github/Anvesha-sonkar04/doc_qa_bot/blob/main/doc_qa_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
os.makedirs(f'{BASE}/data/pdfs', exist_ok=True)
print(os.listdir(f'{BASE}/data'))

['pdfs']


In [5]:
print(os.listdir(f'{BASE}/data/pdfs'))

['3. FiniteAutomata_2.pdf', 'new_syllabus.pdf']


In [6]:
import os
BASE = '/content/drive/MyDrive/doc_qa_bot'

folders = ['data/pdfs', 'index', 'results', 'app', 'notebooks']
for f in folders:
    os.makedirs(f'{BASE}/{f}', exist_ok=True)

# placeholder files
for fname in ['README.md', 'requirements.txt']:
    path = f'{BASE}/{fname}'
    if not os.path.exists(path):
        open(path, 'w').close()

for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    print('  ' * level + os.path.basename(root) + '/')
    for fl in files:
        print('  ' * (level + 1) + fl)

doc_qa_bot/
  doc_qa_bot.ipynb
  README.md
  requirements.txt
  data/
    pdfs/
      3. FiniteAutomata_2.pdf
      new_syllabus.pdf
  app/
  index/
  results/
  notebooks/


In [11]:
!pip install -q sentence-transformers faiss-cpu pypdf

In [8]:
from pypdf import PdfReader

pdf_dir = f'{BASE}/data/pdfs'
for fname in os.listdir(pdf_dir):
    if fname.lower().endswith('.pdf'):
        reader = PdfReader(f'{pdf_dir}/{fname}')
        text = reader.pages[0].extract_text() or ''
        print(fname, '| pages:', len(reader.pages), '| chars on page 1:', len(text))

3. FiniteAutomata_2.pdf | pages: 76 | chars on page 1: 652
new_syllabus.pdf | pages: 263 | chars on page 1: 370


In [9]:
import pandas as pd

def chunk_text(text, size=200, overlap=40):
    words = text.split()
    return [' '.join(words[i:i+size]) for i in range(0, len(words), size - overlap)]

rows = []
for fname in os.listdir(pdf_dir):
    if not fname.lower().endswith('.pdf'):
        continue
    reader = PdfReader(f'{pdf_dir}/{fname}')
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        for c in chunk_text(text):
            if len(c.split()) > 20:          # skip tiny fragments
                rows.append({'source': fname, 'page': page_no, 'text': c})

chunks_df = pd.DataFrame(rows)
chunks_df.to_csv(f'{BASE}/data/chunks.csv', index=False)
print(len(chunks_df), 'chunks saved')
chunks_df.head()

622 chunks saved


,source,page,text
0,3. FiniteAutomata_2.pdf,1,1 Deterministic Finite State Automata (DFA) ……...
1,3. FiniteAutomata_2.pdf,2,2 • The finite control can be described by a t...
2,3. FiniteAutomata_2.pdf,3,"3 Note: • Machine is for accepting a language,..."
3,3. FiniteAutomata_2.pdf,4,4 • An equivalent machine to the previous exam...
4,3. FiniteAutomata_2.pdf,5,5 • Example #2: a c c c b accepted q0 q0 q1 q2...


In [14]:
import os, numpy as np, pandas as pd, faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [17]:
import os
BASE = '/content/drive/MyDrive/doc_qa_bot'
PDF_DIR = f'{BASE}/data/pdfs'

print(os.listdir(PDF_DIR))

['3. FiniteAutomata_2.pdf', 'new_syllabus.pdf']


In [18]:
def chunk_text(text, size=150, overlap=30):
    words = text.split()
    return [' '.join(words[i:i+size]) for i in range(0, len(words), size - overlap)]

def load_pdfs(pdf_dir):
    rows = []
    for fname in os.listdir(pdf_dir):
        if not fname.lower().endswith('.pdf'):
            continue
        reader = PdfReader(f'{pdf_dir}/{fname}')
        for page_no, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ''
            for c in chunk_text(text):
                if len(c.split()) > 20:
                    rows.append({'source': fname, 'page': page_no, 'text': c})
    return pd.DataFrame(rows)

embedder = SentenceTransformer('all-MiniLM-L6-v2')

def build_index(chunks_df):
    emb = embedder.encode(chunks_df['text'].tolist(),
                          normalize_embeddings=True, show_progress_bar=True)
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(np.array(emb, dtype='float32'))
    return idx

chunks_df = load_pdfs(PDF_DIR)
index = build_index(chunks_df)
print(len(chunks_df), 'chunks indexed')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

761 chunks indexed


In [19]:
def search(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(np.array(q, dtype='float32'), k)
    return [{'score': round(float(s), 3), **chunks_df.iloc[i].to_dict()}
            for i, s in zip(ids[0], scores[0])]

In [26]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained('google/flan-t5-large')
gen_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')

def generate(prompt, max_new_tokens=120):
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=512)
    out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tok.decode(out[0], skip_special_tokens=True)

def ask(question, k=3):
    hits = search(question, k)
    context = '\n'.join(h['text'] for h in hits)
    prompt = (f"Read the context and answer the question in one or two complete sentences.\n\n"
              f"Context: {context}\n\nQuestion: {question}\n\n"
              f"Detailed answer:")
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=512)
    out = gen_model.generate(**inputs, max_new_tokens=150,
                             min_new_tokens=25, num_beams=4,
                             no_repeat_ngram_size=3)
    answer = tok.decode(out[0], skip_special_tokens=True)
    print('Answer:', answer, '\n')
    print('Sources:')
    for h in hits:
        print(f"  - {h['source']}, page {h['page']} (score {h['score']})")
    return answer, hits

ask("What Is Air Made Of?")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Answer: The atmosphere is held in place by gravity and moves with the planet as air currents and weather systems develop. Although it becomes thinner with increasing altitude, it has no single sharp upper boundary. Earth’s atmosphere makes life possible by supplying gases needed by living things, moderating temperature, and helping protect the surface from harmful solar radiation and many small objects from space. 

Sources:
  - Atmosphere.pdf, page 1 (score 0.682)
  - Atmosphere.pdf, page 1 (score 0.317)
  - Atmosphere.pdf, page 1 (score 0.263)


('The atmosphere is held in place by gravity and moves with the planet as air currents and weather systems develop. Although it becomes thinner with increasing altitude, it has no single sharp upper boundary. Earth’s atmosphere makes life possible by supplying gases needed by living things, moderating temperature, and helping protect the surface from harmful solar radiation and many small objects from space.',
 [{'score': 0.682,
   'source': 'Atmosphere.pdf',
   'page': 1,
   'text': "Page 1 THE ATMOSPHERE A clear introduction to Earth's blanket of gases 1. What Is the Atmosphere? The atmosphere is the layer of gases surrounding Earth. It is held in place by gravity and moves with the planet as air currents and weather systems develop. Although it becomes thinner with increasing altitude, it has no single sharp upper boundary. Earth’s atmosphere makes life possible by supplying gases needed by living things, moderating temperature, and helping protect the surface from harmful solar rad

# I have replaced the pdf and now i am indexing rechunking them again


In [23]:
print(os.listdir(PDF_DIR))             # confirm the new files are visible

chunks_df = load_pdfs(PDF_DIR)         # re-read all PDFs
index = build_index(chunks_df)         # re-create the embeddings and index
print(len(chunks_df), 'chunks indexed')

# save the new index (overwrites the old one)
faiss.write_index(index, f'{BASE}/index/faiss.index')
chunks_df.to_pickle(f'{BASE}/index/chunks.pkl')

['Plants.pdf', 'Atmosphere.pdf']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

10 chunks indexed
